# Milestone 6C: Distributed Machine Learning with PySpark MLlib

This notebook trains and benchmarks distributed classification models for AML money mule detection using **pyspark.ml**.

### Models Evaluated:
- **Spark Gradient-Boosted Trees (GBTClassifier)**
- **Spark Random Forest (RandomForestClassifier)**
- **Spark Regularized Logistic Regression (LogisticRegression)**

### Key Highlights:
- `VectorAssembler` and `StandardScaler` feature preprocessing pipeline.
- Evaluated with **Precision-Recall AUC (PR-AUC)** via `BinaryClassificationEvaluator`.
- Cross-model comparison against the single-node XGBoost baseline and PyG GraphSAGE GNN.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.spark_ml_pipeline import create_spark_session, SparkMLMuleClassifier

spark = create_spark_session('Databricks-M6C-MLlib')

features_path = '../data/processed/account_features.csv'
split_path = '../data/processed/split_accounts.csv'

raw_features = spark.read.option('header', 'true').option('inferSchema', 'true').csv(features_path)
split_df = spark.read.option('header', 'true').csv(split_path)

classifier = SparkMLMuleClassifier(spark)
train_df, val_df, test_df = classifier.prepare_data(raw_features, split_df)


C:\Projects\Mule-tiplier Extended\chain-reaction-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


2026-10-29 12:08:59,972 [INFO] src.spark_ml_pipeline — Partitioned data using split_accounts mapping.


2026-10-29 12:09:05,867 [INFO] src.spark_ml_pipeline — Splits ready — Train: 360556, Val: 51508, Test: 103016


## 1. Distributed Model Training Suite
We train GBT, Random Forest, and Logistic Regression on distributed partitions.


In [2]:
# Train models on sample for fast demonstration
sample_train = train_df.sample(False, 0.25, seed=42)
sample_val = val_df.sample(False, 0.50, seed=42)
sample_test = test_df.sample(False, 0.50, seed=42)

results = classifier.run_benchmark_suite(sample_train, sample_val, sample_test)
classifier.print_comparison_table(results)


2026-10-29 12:09:05,889 [INFO] src.spark_ml_pipeline — Training Spark MLlib GBTClassifier...


2026-10-29 12:09:28,434 [INFO] src.spark_ml_pipeline — GBT Complete — Train PR-AUC: 0.1311, Val: 0.0781, Test: 0.0762


2026-10-29 12:09:28,435 [INFO] src.spark_ml_pipeline — Training Spark MLlib RandomForestClassifier...


2026-10-29 12:09:34,422 [INFO] src.spark_ml_pipeline — Random Forest Complete — Test PR-AUC: 0.0626


2026-10-29 12:09:34,424 [INFO] src.spark_ml_pipeline — Training Spark MLlib LogisticRegression...


2026-10-29 12:09:40,573 [INFO] src.spark_ml_pipeline — Logistic Regression Complete — Test PR-AUC: 0.0352



HEAD-TO-HEAD MODEL BENCHMARK COMPARISON TABLE
Model Architecture             | Framework        | Val PR-AUC   | Test PR-AUC 
--------------------------------------------------------------------------------
Random Guessing Floor          | Theoretical      | 0.0123       | 0.0123      
XGBoost (Milestone 2)          | xgboost/sklearn  | —            | 0.0914      
XGBoost (Milestone 3)          | xgboost/sklearn  | 0.0855       | 0.0834      
GraphSAGE GNN (Milestone 3)    | PyTorch Geometric | 0.2091       | 0.1884      
Spark GBT                      | pyspark.ml       | 0.0781       | 0.0762      
Spark Random Forest            | pyspark.ml       | 0.0640       | 0.0626      
Spark Logistic Regression      | pyspark.ml       | 0.0365       | 0.0352      



## 2. Model Persistence & Governance
We save the trained Spark ML pipeline to disk for production serving.


In [3]:
best_gbt = [r for r in results if r.model_name == 'Spark GBT'][0]
classifier.save_model(best_gbt.model_instance, '../outputs/models/spark_gbt_pipeline')
print('Spark MLlib Pipeline saved successfully.')
spark.stop()


2026-10-29 12:09:46,201 [INFO] src.spark_ml_pipeline — Saved Spark ML model pipeline to '../outputs/models/spark_gbt_pipeline'


Spark MLlib Pipeline saved successfully.
